# VentureScope — Integrated Research Pipeline
**Run cells in order.** This notebook prepares four distinct evidence sources, evaluates supervised sentiment baselines, constructs domain-specific indicators, and powers the Streamlit dashboard. It does not invent current demand, human labels, or validation outcomes.

In [ ]:
from pathlib import Path
import sys, json, warnings
DATA_DIR = Path(r"D:\Downloadeds")  # Change only if your ZIP files are elsewhere
PROJECT_DIR = Path.cwd()
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
assert (PROJECT_DIR / "src").exists(), "Open this notebook from the extracted VentureScope_Complete folder"
sys.path.insert(0,str(PROJECT_DIR))
print("Dataset folder:",DATA_DIR,"exists:",DATA_DIR.exists())
print("Project:",PROJECT_DIR)
print("ZIP files:",[p.name for p in DATA_DIR.glob('*.zip')])

## 1. Archive inventory and verified format

In [ ]:
from src.pipeline import archive,inventory,amazon,olist,uci,audit,build_olist_indicators,build_uci_indicators,build_amazon_indicators,screening,save_csv
for key in ['amazon_reviews','amazon_products','olist','uci']:
    path=archive(DATA_DIR,key)
    print('\n',key,path.name)
    for name,size in inventory(path)[:25]:print(' ',name,round(size/1024**2,2),'MB')

## 2. Load Amazon Review Polarity and Amazon Products
The review file is headerless CSV; labels 1/2 are sentiment. It has no ASIN or category ID. No product-level join is attempted.

In [ ]:
amazon_train,amazon_test,amazon_products,amazon_categories=amazon(DATA_DIR,review_rows=80000,product_rows=150000)
for name,df in [('Amazon train',amazon_train),('Amazon test',amazon_test),('Amazon products',amazon_products),('Amazon categories',amazon_categories)]:
    print(audit(name,df));display(df.head(3))
assert set(amazon_train.sentiment_label.unique()).issubset({1,2})
assert 'category_name' in amazon_products

## 3. Load Olist transaction tables

In [ ]:
olist_tables=olist(DATA_DIR)
for name,df in olist_tables.items():print(audit('Olist '+name,df))
olist_joined,olist_monthly,olist_categories=build_olist_indicators(olist_tables)
display(olist_categories.sort_values('orders',ascending=False).head(12))

## 4. Load UCI Retail
Bounded to 350,000 rows to keep notebook memory manageable; the manifest explicitly records sample limits.

In [ ]:
uci_raw=uci(DATA_DIR,max_rows=350000)
print(audit('UCI Retail',uci_raw));display(uci_raw.head(3))
uci_clean,uci_monthly,uci_stats=build_uci_indicators(uci_raw)
print(uci_stats);display(uci_monthly.tail())

## 5. Dataset profiling and provenance

In [ ]:
import pandas as pd,numpy as np
manifest={'data_dir':str(DATA_DIR),'archives':{k:[{'member':name,'bytes':size} for name,size in inventory(archive(DATA_DIR,k))] for k in ['amazon_reviews','amazon_products','olist','uci']},'sample_limits':{'amazon_train':80000,'amazon_test':20000,'amazon_products':150000,'uci':350000},'counts':{'amazon_train':len(amazon_train),'amazon_test':len(amazon_test),'amazon_products':len(amazon_products),'olist_order_items':len(olist_joined),'uci':len(uci_raw)},'caveats':['Amazon reviews are polarity-labelled without ASIN/category','Amazon product data is a sampled snapshot','Olist Brazil and UCI UK are historical and cannot measure 2026 local market']}
(OUTPUT_DIR/'manifest.json').write_text(json.dumps(manifest,indent=2,default=str))
print(json.dumps(manifest['counts'],indent=2))

## 6. Amazon category demand and competition proxies

In [ ]:
amazon_indicators=build_amazon_indicators(amazon_products)
from src.assessment import category_scores,market_score,readiness,recommendations,sensitivity,founder_invariance_test,sample_assessment
amazon_scores=category_scores(amazon_indicators,olist_categories)
display(amazon_scores.sort_values('bought_last_month_sum',ascending=False).head(15))
save_csv(amazon_scores,OUTPUT_DIR/'amazon_category_indicators.csv')

## 7. Historical Olist demand, competition and logistics

In [ ]:
save_csv(olist_monthly,OUTPUT_DIR/'olist_monthly.csv')
save_csv(olist_categories,OUTPUT_DIR/'olist_category_indicators.csv')
if 'order_reviews' in olist_tables:
    olist_review=olist_tables['order_reviews'].copy()
    print('Olist review columns:',olist_review.columns.tolist())
    if 'review_score' in olist_review:
        print('Olist review score distribution:');display(olist_review.review_score.value_counts(dropna=False).sort_index())
    if 'review_comment_message' in olist_review:
        save_csv(olist_review[['review_id','review_score','review_comment_message']].head(20000),OUTPUT_DIR/'olist_review_sample.csv')
if 'orders' in olist_tables:
    o=olist_tables['orders'].copy()
    for c in ['order_delivered_customer_date','order_estimated_delivery_date']:
        if c in o:o[c]=pd.to_datetime(o[c],errors='coerce')
    if {'order_delivered_customer_date','order_estimated_delivery_date'}.issubset(o.columns):
        valid=o.dropna(subset=['order_delivered_customer_date','order_estimated_delivery_date'])
        print('Historic Olist late-delivery fraction:',round((valid.order_delivered_customer_date>valid.order_estimated_delivery_date).mean(),4))

## 8. Historical UCI demand and repeat purchasing

In [ ]:
save_csv(uci_monthly,OUTPUT_DIR/'uci_monthly.csv')
(OUTPUT_DIR/'uci_stats.json').write_text(json.dumps(uci_stats,indent=2,default=str))
display(uci_monthly)

## 9. Customer-problem candidates — not ground truth

In [ ]:
screened,rules=screening(amazon_train)
print('Candidate phrase rules:',rules)
print('Candidate problem fraction:',screened.candidate_problem.mean())
display(screened.loc[screened.candidate_problem,['review_title','review_text','candidate_problem']].head(10))
save_csv(screened[['review_title','review_text','sentiment_label','candidate_problem']].head(10000),OUTPUT_DIR/'review_screening_candidates.csv')

## 10. Prepare an annotation sheet
Annotate `complaint`, `feature_request`, `unmet_need`, or `none` manually. These labels must not be inferred from polarity scores.

In [ ]:
annotation=screened[['review_title','review_text','sentiment_label','candidate_problem']].sample(n=min(400,len(screened)),random_state=42).copy()
annotation.insert(0,'annotation_id',range(1,len(annotation)+1))
annotation['human_label']=''
annotation['severity']=''
annotation['annotator_id']=''
save_csv(annotation,OUTPUT_DIR/'human_annotation_template.csv')
print('Human annotation template created:',len(annotation),'rows')

## 11. Supervised NLP evaluation: sentiment only
The original predefined train/test partitions are respected. These scores must **not** be reported as unmet-need detection performance.

In [ ]:
from src.models import train_sentiment,time_backtest
sentiment_results=train_sentiment(amazon_train,amazon_test,OUTPUT_DIR/'models')
display(sentiment_results)
save_csv(sentiment_results,OUTPUT_DIR/'sentiment_model_metrics.csv')

## 12. Historical Olist demand forecast backtest
Compares last-month naive and rolling-three-month forecasts on the final three available months for categories with sufficient history.

In [ ]:
forecast_results=time_backtest(olist_monthly,OUTPUT_DIR)
print('Evaluated categories:',forecast_results.category.nunique() if len(forecast_results) else 0)
display(forecast_results.head(20))
save_csv(forecast_results,OUTPUT_DIR/'olist_forecast_backtest.csv')

## 13. Visualizations

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(10,5))
plot=amazon_scores.nlargest(12,'bought_last_month_sum').sort_values('bought_last_month_sum')
ax.barh(plot.category,plot.bought_last_month_sum)
ax.set_title('Amazon product sample: reported bought-in-last-month totals')
ax.set_xlabel('Sum of reported product-level counts (sample only)')
fig.tight_layout();fig.savefig(OUTPUT_DIR/'amazon_demand_proxy.png',dpi=150);plt.show()
fig,ax=plt.subplots(figsize=(10,4));ax.plot(uci_monthly.month,uci_monthly.revenue,marker='o');ax.tick_params(axis='x',rotation=60);ax.set_title('UCI historic monthly sample revenue');fig.tight_layout();fig.savefig(OUTPUT_DIR/'uci_historical_revenue.png',dpi=150);plt.show()

## 14. Controlled domain evidence and source reconciliation
Category names across Amazon, Olist and UCI are **not equivalent**. Export mapping candidates for manual verification; no automatic record-level merge.

In [ ]:
mapping=pd.DataFrame({'amazon_category':sorted(amazon_scores.category.astype(str).unique())})
mapping['controlled_domain']=''
mapping['mapping_verified_by']=''
save_csv(mapping,OUTPUT_DIR/'domain_mapping_review_template.csv')
print('Mapping review rows:',len(mapping))
print('Olist categories:',olist_categories.category.nunique())

## 15. Assessment with strict evidence gate
A domain-matched validated complaint/need rate, product fit, and accessibility are not available in these archives. Consequently, the prototype **must** return insufficient evidence rather than inventing an opportunity score.

In [ ]:
chosen=amazon_scores.sort_values('products',ascending=False).iloc[0]
founder={'budget_readiness':0.3,'team_readiness':0.4,'prototype':False,'early_users':False}
assessment=sample_assessment(chosen)
print('Example domain:',chosen.category)
print('Assessment:',json.dumps(assessment,indent=2,default=str))
print('Founder readiness (separate):',readiness(founder))
display(recommendations(assessment,founder))
assert assessment['score'] is None
assert founder_invariance_test(assessment['dimensions'],assessment['provenance'])

## 16. Scenario and rubric sensitivity with **explicitly illustrative** values
This tests the mechanics only. It is **not** a market finding and is never exported as observed evidence.

In [ ]:
example_dimensions={'market_demand':65,'customer_problem':70,'competition':45,'product_fit':75,'accessibility':55}
example_coverage={k:1.0 for k in example_dimensions}
example=market_score(example_dimensions,example_coverage)
print('ILLUSTRATIVE / SYNTHETIC TEST ONLY:',example)
print('Weight sensitivity:',sensitivity(example_dimensions,example_coverage,n=1000))
assert founder_invariance_test(example_dimensions,example_coverage)
print('PASS: founder input does not change market assessment')

## 17. Export research artifacts

In [ ]:
exports={'amazon_category_indicators':amazon_scores,'olist_category_indicators':olist_categories,'olist_monthly':olist_monthly,'uci_monthly':uci_monthly,'sentiment_model_metrics':sentiment_results,'olist_forecast_backtest':forecast_results}
for name,df in exports.items():save_csv(df,OUTPUT_DIR/(name+'.csv'))
with pd.ExcelWriter(OUTPUT_DIR/'VentureScope_Results.xlsx',engine='openpyxl') as writer:
    for name,df in exports.items():df.head(50000).to_excel(writer,sheet_name=name[:31],index=False)
(OUTPUT_DIR/'assessment_example.json').write_text(json.dumps({'domain':str(chosen.category),'assessment':assessment,'founder_readiness':readiness(founder)},indent=2,default=str))
print('Created files:')
for f in sorted(OUTPUT_DIR.rglob('*')):
    if f.is_file():print(' ',f.relative_to(PROJECT_DIR))

## 18. Dashboard and next evidence gates
From a terminal in the project folder run: `python -m streamlit run app.py`.

**Research validity:** sentiment metrics and historical forecast backtests are real if this notebook ran. Aspect-level unmet-need performance requires the annotation worksheet to be labelled and evaluated. A current regional market assessment requires matching recent evidence; external experts and user-study results cannot be generated from these four ZIPs.

## Integrated business idea → category → evidence → assessment
These cells use observed Amazon product evidence, leave unsupported dimensions unscored, and provide a human-annotation workflow. The earlier example scores are **illustrative**, not empirical outcomes.

In [ ]:
from src.concept import match_categories, competitor_profile, evidence_report
BUSINESS_IDEA = "Affordable durable luggage for frequent travelers"  # Change this to your own idea
matches = match_categories(BUSINESS_IDEA, amazon_products['category_name'])
display(matches)
assert len(matches)>0, "No categories were available"
# A human must confirm the category. This selection is only a candidate.
SELECTED_CATEGORY = matches.iloc[0]['category']
print('Candidate category:', SELECTED_CATEGORY)
profile, competitors = competitor_profile(amazon_products, SELECTED_CATEGORY)
print('Product evidence:', profile)
display(competitors.head(10))
save_csv(amazon_products, OUTPUT_DIR/'amazon_products_sample_with_categories.csv')


## Human-labelled customer-problem dataset
Fill `label_primary` with one of `complaint`, `unmet_need`, `feature_request`, `none`. A second person can fill `label_second` for agreement. **Do not use the sentiment label as a need label.** The evaluation runs only when enough real annotations exist.

In [ ]:
from src.annotation import prepare_annotation_sheet, evaluate_annotations
annotation_path=OUTPUT_DIR/'customer_problem_annotation.csv'
if not annotation_path.exists():
    annotation_sheet=prepare_annotation_sheet(amazon_train,n=600)
    annotation_sheet.to_csv(annotation_path,index=False)
    print('Created annotation worksheet:',annotation_path)
else:
    print('Existing annotations preserved:',annotation_path)
annotation_status=evaluate_annotations(annotation_path,OUTPUT_DIR)
print(annotation_status)


## Historical demand baseline comparison
This compares historical forecasting baselines using chronological splits. It does **not** forecast current UK, Brazilian or Pakistani markets.

In [ ]:
from src.historical import forecast_backtest
historical_results=forecast_backtest(olist_monthly, value_col='orders')
print('Evaluated historical categories:',historical_results.category.nunique() if len(historical_results) else 0)
display(historical_results.sort_values('sMAPE_percent').head(15) if len(historical_results) else historical_results)
save_csv(historical_results,OUTPUT_DIR/'historical_forecast_baselines.csv')


## Actual assessment for the selected concept
No need/fit/accessibility scores are invented. The system correctly returns **Insufficient evidence** until those dimensions have defensible inputs.

In [ ]:
from src.assessment import sample_assessment, readiness, recommendations, founder_invariance_test
matched=amazon_scores[amazon_scores.category==SELECTED_CATEGORY]
if len(matched):
    selected=matched.iloc[0]
    founder={'budget_readiness':.3,'team_readiness':.4,'prototype':False,'early_users':False}
    actual_assessment=sample_assessment(selected)
    actual_recommendations=recommendations(actual_assessment,founder)
    assert founder_invariance_test(actual_assessment['dimensions'],actual_assessment['provenance'])
    complete_report=evidence_report(BUSINESS_IDEA,SELECTED_CATEGORY,profile,actual_assessment,readiness(founder),actual_recommendations.to_dict('records'))
    (OUTPUT_DIR/'concept_assessment.json').write_text(json.dumps(complete_report,indent=2,default=str),encoding='utf-8')
    print('Assessment:',actual_assessment['category'],'| Coverage:',actual_assessment['coverage'])
    display(actual_recommendations)
else:
    print('Selected category missing from category indicators; inspect the mapping')


## Research completion gates
1. Verify and document source licences. 2. Annotate reviews and evaluate need classifiers against the majority baseline. 3. Confirm taxonomy mapping manually. 4. Collect recent, geographically matched evidence before assigning current-market grades. 5. Run retrospective backtesting, expert and user studies where approvals and participants permit. Do not report unperformed experiments as results.